# Task 1: Use Linear Regression to predict Q

### Subtask 1: Fit Linear Regression model to relevant variables (n , P , Δp , and others) and predict Q (flow)
### Subtask 2: Report test error (MSE)

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupKFold, train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV, RidgeCV, LinearRegression
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.metrics import mean_squared_error, root_mean_squared_error, mean_absolute_error, r2_score
from sklearn.pipeline import Pipeline

In [18]:
# load data for case 1
df_train1 = pd.read_csv('../data/case1/train1.csv')
df_test1 = pd.read_csv('../data/case1/test1.csv')

for df in [df_train1, df_test1]:
    # change to binary 
    mapping = {
        "stable": 0,
        "transient": 1
    }
    df['operating_state'] = df['operating_state'].map(mapping)

    # speed in krpm, n
    df['krpm'] = df['speed_rpm'] / 1000

    # pressure rise across pump
    df['delta_P'] = (
        df['outlet_pressure_bar']
        - df['inlet_pressure_bar']
    )

    # speed terms
    df['n2'] = df['krpm']**2
    df['n3'] = df['krpm']**3

    # power terms
    df['P2'] = df['power_w']**2
    df['P3'] = df['power_w']**3

    # delta terms
    df['delta_P2'] = df['delta_P']**2
    df['delta_P3'] = df['delta_P']**3

    # interaction terms
    df['nP'] = df['krpm'] * df['power_w']
    df['n_delta_P'] = df['krpm'] * df['delta_P']
    df['P_delta_P'] = df['power_w'] * df['delta_P']


In [19]:
# model 1_1 – only original features

X_1 = df_train1[
    [
        'krpm',
        'power_w',
        'delta_P'
    ]
]

y_1 = df_train1['flow_m3_h']

# KFold since we only have one pump_id
kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

model_1 = LinearRegression()

scores = cross_val_score(
    model_1,
    X_1,
    y_1,
    cv=kf,
    scoring="neg_root_mean_squared_error"
)

rmse_scores_1 = -scores

print("Fold RMSEs:", rmse_scores_1)
print("Average RMSE:", rmse_scores_1.mean())

Fold RMSEs: [3.55782366 3.62848508 3.48484666 3.1726527  3.55629547]
Average RMSE: 3.4800207140940778


In [20]:
# model 1_2 – original + polynomial

X_2 = df_train1[
    [
        'krpm',
        'power_w',
        'delta_P',

        # speed terms
        'n2',
        'n3',
    
        # power terms
        'P2',
        'P3',
    
        # delta terms
        'delta_P2',
        'delta_P3'
    ]
]

y_2 = df_train1['flow_m3_h']

# KFold since we only have one pump_id
kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

model_2 = LinearRegression()

scores = cross_val_score(
    model_2,
    X_2,
    y_2,
    cv=kf,
    scoring="neg_root_mean_squared_error"
)

rmse_scores_2 = -scores

print("Fold RMSEs:", rmse_scores_2)
print("Average RMSE:", rmse_scores_2.mean())

Fold RMSEs: [1.60272568 1.54913639 1.58679329 1.51360472 1.49443878]
Average RMSE: 1.5493397706612544


In [31]:
# model 1_3 – original + polynomial + interaction features

X_3 = df_train1[
    [
        'krpm',
        'power_w',
        'delta_P',
        
        # speed terms
        'n2',
    
        # power terms
        'P2',
    
        # delta terms
        'delta_P2',

        # interaction terms
        'nP',
        'n_delta_P',
        'P_delta_P'
    ]
]

y_3 = df_train1['flow_m3_h']

# KFold since we only have one pump_id
kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

model_3 = LinearRegression()

scores = cross_val_score(
    model_3,
    X_3,
    y_3,
    cv=kf,
    scoring="neg_root_mean_squared_error"
)

rmse_scores_3 = -scores

print("Fold RMSEs:", rmse_scores_3)
print("Average RMSE:", rmse_scores_3.mean())

Fold RMSEs: [1.29743527 1.29444548 1.27424397 1.15011008 1.15752873]
Average RMSE: 1.2347527052761031


### Best model: model 1_3 – original + polynomial + interaction features

In [33]:
best_model = LinearRegression()

best_model.fit(X_3, y_3)

X_test = df_test1[
    [
        'krpm',
        'power_w',
        'delta_P',
        'n2',
        'P2',
        'delta_P2',
        'nP',
        'n_delta_P',
        'P_delta_P'
    ]
]

y_test = df_test1['flow_m3_h']

y_1_pred = best_model.predict(X_test)

mse_test = round(mean_squared_error(y_true = y_test, y_pred = y_1_pred) , 5)
print(f"Test MSE: {mse_test}")

rmse_test = round(root_mean_squared_error(y_test, y_1_pred), 5)
print(f"Test RMSE: {rmse_test}")

mae_test = mean_absolute_error(y_test, y_1_pred)
print(f"Test MAE: {mae_test:.5f}")


Test MSE: 1.2419
Test RMSE: 1.11441
Test MAE: 0.82660


In [7]:
# exclude nonimportant columns and define X and y
exclude_cols = [
    'pump_id',
    'speed_rpm',
    'elapsed_s',
    'operating_state',
    'flow_m3_h',
    'inlet_pressure_bar',
    'outlet_pressure_bar',
]

In [8]:
df_train1

,pump_id,elapsed_s,operating_state,speed_rpm,power_w,inlet_pressure_bar,outlet_pressure_bar,flow_m3_h,krpm,delta_P,n2,n3,P2,P3,delta_P2,delta_P3,nP,n_delta_P,P_delta_P
0,99223468_model_01,14.5,0,4230.396017,669.992714,1.057136,2.414427,0.652884,4.230396,1.357291,17.896250,75.708227,4.488902e+05,3.007532e+08,1.842239,2.500454,2834.334510,5.741878,909.375057
1,99223468_model_01,24.5,0,3796.519900,931.528480,2.097392,2.390436,43.040564,3.796520,0.293043,14.413563,54.721380,8.677453e+05,8.083295e+08,0.085874,0.025165,3536.566413,1.112545,272.978224
2,99223468_model_01,34.5,0,3508.264375,724.198083,2.247896,2.433662,40.939560,3.508264,0.185766,12.307919,43.179433,5.244629e+05,3.798150e+08,0.034509,0.006411,2540.678334,0.651717,134.531512
3,99223468_model_01,44.5,0,1498.405243,99.463217,2.287246,2.451128,11.857951,1.498405,0.163881,2.245218,3.364247,9.892932e+03,9.839828e+05,0.026857,0.004401,149.036206,0.245561,16.300180
4,99223468_model_01,64.5,0,3224.873653,608.195308,2.464618,2.696299,36.000996,3.224874,0.231682,10.399810,33.538074,3.699015e+05,2.249724e+08,0.053676,0.012436,1961.353025,0.747144,140.907615
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
475,99223468_model_01,5944.5,0,4376.002303,815.229829,1.016499,2.478924,3.032111,4.376002,1.462426,19.149396,83.797802,6.645997e+05,5.418015e+08,2.138689,3.127673,3567.447611,6.399578,1192.212957
476,99223468_model_01,5964.5,0,3514.587072,515.961500,1.411002,2.338274,5.566221,3.514587,0.927272,12.352322,43.413312,2.662163e+05,1.373573e+08,0.859833,0.797298,1813.391617,3.258977,478.436423
477,99223468_model_01,5974.5,0,2935.129740,534.173942,2.178489,2.678476,22.873572,2.935130,0.499988,8.614987,25.286103,2.853418e+05,1.524222e+08,0.249988,0.124991,1567.869824,1.467528,267.080327
478,99223468_model_01,5984.5,0,2214.152995,154.677482,2.032066,2.410823,3.776250,2.214153,0.378757,4.902473,10.854826,2.392512e+04,3.700678e+06,0.143457,0.054335,342.479610,0.838627,58.585233


In [9]:
df_train1["pump_id"].nunique()
df_train1["pump_id"].value_counts().head()

pump_id
99223468_model_01    480
Name: count, dtype: int64

### Subtask 3: Report parameters of best-performing Linear Regression model

In [34]:
# parameters for model_1_3

print(f"Intercept:, {best_model.intercept_:.3f}")

for feature, coef in zip(X_test.columns, best_model.coef_):
    print(f"{feature}, {coef:.3f}")

Intercept:, -17.941
krpm, 27.934
power_w, 0.122
delta_P, -98.981
n2, -6.334
P2, 0.000
delta_P2, 25.822
nP, -0.018
n_delta_P, 16.605
P_delta_P, -0.035


In [35]:
print(best_model.coef_)


[ 2.79338966e+01  1.22038655e-01 -9.89809507e+01 -6.33394869e+00
  1.45413805e-05  2.58219730e+01 -1.77209296e-02  1.66049743e+01
 -3.53875620e-02]


Predicted flow is:

$$\hat{Q}= -17.941+27.934(krpm)+0.122(power_w)-98.981(\nabla P)-6.334(n^2)+1.45 \times 10^{-5}(p^2)+25.822(\nabla P^2)-0.018(nP)+16.605(n\nabla P)-0.035(P \nabla P)$$

Reduced model removing third-order polynomial $p^3, n^3, delta_P^3$ gives an increase of 2.2% RMSE suggesting that lower order terms have more predictive power.